In [10]:
import pandas as pd
import re
import os
df = pd.read_csv("../data/FeynmanEquations (1).csv",
                 encoding='utf-8-sig', skipinitialspace=True)
df.columns = [c.strip() for c in df.columns]

# just look at first 3 rows, key columns
df[['Filename', 'Formula', '# variables',
    'v1_name','v1_low','v1_high',
    'v2_name','v2_low','v2_high']].head(3)

,Filename,Formula,# variables,v1_name,v1_low,v1_high,v2_name,v2_low,v2_high
0,I.6.2a,exp(-theta**2/2)/sqrt(2*pi) ...,1.0,theta,1.0,3.0,NaN,NaN,NaN
1,I.6.2,exp(-(theta/sigma)**2/2)/(sqrt(2*pi)*sigma) ...,2.0,sigma,1.0,3.0,theta,1.0,3.0
2,I.6.2b,exp(-((theta-theta1)/sigma)**2/2)/(sqrt(2*pi)*...,3.0,sigma,1.0,3.0,theta,1.0,3.0


In [11]:
SYMBOL_TO_NAME = {
    '+' : 'ADD',
    '-' : 'SUB',
    '*' : 'MUL',
    '/' : 'DIV',
    '**': 'POW',     # original Python notation
    '^' : 'POW',     # after grammar conversion
    'neg': 'NEG',    # unary minus
}

# --- Decoding: name → symbol  (used when testing model outputs) ---
NAME_TO_SYMBOL = {
    'ADD': '+',
    'SUB': '-',
    'MUL': '*',
    'DIV': '/',
    'POW': '**',
    'NEG': 'neg',    # unary minus (no direct symbol)
}

# --- Functions recognized in the grammar ---
FUNCTION_NAMES = {'exp', 'sqrt', 'sin', 'cos', 'tan',
                  'arcsin', 'arccos', 'arctan', 'tanh', 'ln', 'log'}

# --- Full grammar: every token the model can output ---
GRAMMAR = {
    'operators':  list(NAME_TO_SYMBOL.keys()),   # ADD, SUB, MUL, DIV, POW, NEG
    'functions':  sorted(FUNCTION_NAMES),         # exp, sqrt, sin, cos, ...
    'constants':  ['pi'],                         # constants appearing in formulas
}


In [12]:
#  TOKENIZER that labels every piece
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class Token:
    type: str      # NUMBER, FUNC, VAR, OP, LPAREN, RPAREN
    value: str

# these are the function names that appear in the Feynman formulas
FUNCTIONS = {'exp','sqrt','sin','cos','tan','arcsin','arccos','arctan','tanh','ln','log'}

def tokenize(expr: str) -> List[Token]:
    tokens = []
    i = 0
    while i < len(expr):
        ch = expr[i]

        # skip spaces
        if ch == ' ':
            i += 1
            continue

        # numbers like 1, 23, 3.14
        num_match = re.match(r'\d+\.?\d*', expr[i:])
        if num_match:
            tokens.append(Token('NUMBER', num_match.group()))
            i += num_match.end()
            continue

        # words — could be a function name or a variable name
        id_match = re.match(r'[a-zA-Z_]\w*', expr[i:])
        if id_match:
            name = id_match.group()
            if name in FUNCTIONS:
                tokens.append(Token('FUNC', name))    # label it as FUNC
            else:
                tokens.append(Token('VAR', name))      # label it as VAR
            i += id_match.end()
            continue

        # single-character operators
        if ch in '+-*/^':
            tokens.append(Token('OP', ch))
            i += 1
            continue

        # parentheses
        if ch == '(':
            tokens.append(Token('LPAREN', '('))
            i += 1
            continue
        if ch == ')':
            tokens.append(Token('RPAREN', ')'))
            i += 1
            continue

        raise ValueError(f"Unexpected character '{ch}' at position {i}")

    return tokens



In [13]:
@dataclass
class NumNode:
    """A number like 2, 3, 4"""
    value: str

@dataclass
class VarNode:
    """A variable like theta, m_0, pi"""
    name: str

@dataclass
class BinOpNode:
    """A binary operation:  left OP right    e.g. theta ^ 2"""
    op: str
    left: object
    right: object

@dataclass
class UnaryNegNode:
    """Unary negation:  -something"""
    operand: object

@dataclass
class FuncNode:
    """A function call:  func(arg)    e.g. sin(theta)"""
    name: str
    arg: object


# --- The parser ---

class Parser:
    def __init__(self, tokens: List[Token]):
        self.tokens = tokens
        self.pos = 0

    def peek(self) -> Optional[Token]:
        return self.tokens[self.pos] if self.pos < len(self.tokens) else None

    def consume(self) -> Token:
        tok = self.tokens[self.pos]
        self.pos += 1
        return tok

    def expect(self, ttype: str) -> Token:
        tok = self.consume()
        if tok.type != ttype:
            raise ValueError(f"Expected {ttype}, got {tok}")
        return tok

    def parse(self):
        node = self.parse_expression()
        if self.pos < len(self.tokens):
            raise ValueError(f"Leftover token: {self.tokens[self.pos]}")
        return node

    # Level 1 (lowest precedence): + and -
    def parse_expression(self):
        left = self.parse_term()
        while self.peek() and self.peek().type == 'OP' and self.peek().value in ('+', '-'):
            op = self.consume().value
            right = self.parse_term()
            left = BinOpNode(op, left, right)
        return left

    # Level 2: * and /
    def parse_term(self):
        left = self.parse_power()
        while self.peek() and self.peek().type == 'OP' and self.peek().value in ('*', '/'):
            op = self.consume().value
            right = self.parse_power()
            left = BinOpNode(op, left, right)
        return left

    # Level 3: ^  (right-associative: a^b^c = a^(b^c))
    def parse_power(self):
        base = self.parse_unary()
        if self.peek() and self.peek().type == 'OP' and self.peek().value == '^':
            self.consume()
            exp = self.parse_power()       # ← calls itself = right-associative
            return BinOpNode('^', base, exp)
        return base

    # Level 4: unary minus  (-x)
    def parse_unary(self):
        if self.peek() and self.peek().type == 'OP' and self.peek().value == '-':
            self.consume()
            operand = self.parse_unary()
            return UnaryNegNode(operand)
        if self.peek() and self.peek().type == 'OP' and self.peek().value == '+':
            self.consume()
            return self.parse_unary()
        return self.parse_atom()

    # Level 5 (highest): numbers, variables, function calls, (grouped)
    def parse_atom(self):
        tok = self.peek()
        if tok is None:
            raise ValueError("Unexpected end of expression")

        if tok.type == 'NUMBER':
            self.consume()
            return NumNode(tok.value)

        if tok.type == 'FUNC':          # function call: name ( expression )
            self.consume()
            self.expect('LPAREN')
            arg = self.parse_expression()
            self.expect('RPAREN')
            return FuncNode(tok.value, arg)

        if tok.type == 'VAR':
            self.consume()
            return VarNode(tok.value)

        if tok.type == 'LPAREN':        # grouped expression: ( expression )
            self.consume()
            node = self.parse_expression()
            self.expect('RPAREN')
            return node

        raise ValueError(f"Unexpected token: {tok}")


In [ ]:
OP_NAMES = {'+': 'ADD', '-': 'SUB', '*': 'MUL', '/': 'DIV', '^': 'POW'}

OPERATORS = {'ADD', 'SUB', 'MUL', 'DIV', 'POW', 'NEG'}
CONSTANTS = {'pi'}

def ast_to_prefix(node) -> str:
    if isinstance(node, NumNode):
        return node.value
    if isinstance(node, VarNode):
        return node.name
    if isinstance(node, BinOpNode):
        name = OP_NAMES.get(node.op, node.op)
        return f"{name} {ast_to_prefix(node.left)} {ast_to_prefix(node.right)}"
    if isinstance(node, UnaryNegNode):
        return f"NEG {ast_to_prefix(node.operand)}"
    if isinstance(node, FuncNode):
        return f"{node.name} {ast_to_prefix(node.arg)}"
    raise ValueError(f"Unknown node: {type(node)}")


def infix_to_named(expr: str) -> str:
    """Replace operator symbols with word names in the infix expression."""
    toks = tokenize(expr)
    parts = []
    for t in toks:
        if t.type == 'OP' and t.value in OP_NAMES:
            parts.append(OP_NAMES[t.value])
        else:
            parts.append(t.value)
    return ' '.join(parts)


def infix_to_prefix(expr: str) -> str:
    tokens = tokenize(expr)
    tree = Parser(tokens).parse()
    return ast_to_prefix(tree)


def classify_prefix_tokens(prefix_expr: str) -> str:
    OP_TAGS = {
        'ADD': 'OP_ADD',
        'SUB': 'OP_SUB',
        'MUL': 'OP_MUL',
        'DIV': 'OP_DIV',
        'POW': 'OP_POW',
        'NEG': 'OP_NEG',
    }

    types = []
    for tok in prefix_expr.split():
        if tok in OP_TAGS:
            types.append(OP_TAGS[tok])
        elif tok in FUNCTIONS:
            types.append('FUNC')
        elif tok in CONSTANTS:
            types.append('CONST')
        elif tok.replace('.', '', 1).isdigit():
            types.append('NUM')
        else:
            types.append('VAR')
    return ' '.join(types)


# --- TEST ---
pf = infix_to_prefix("exp(-theta^2/2)/sqrt(2*pi)")
ct = classify_prefix_tokens(pf)
print("prefix:", pf)
print("types :", ct)
print()

pf2 = infix_to_prefix("G*m1*m2/((x2-x1)^2+(y2-y1)^2)")
ct2 = classify_prefix_tokens(pf2)
print("prefix:", pf2)
print("types :", ct2)

prefix: DIV exp DIV POW NEG theta 2 2 sqrt MUL 2 pi
types : OP_DIV FUNC OP_DIV OP_POW OP_NEG VAR NUM NUM FUNC OP_MUL NUM CONST

prefix: DIV MUL MUL G m1 m2 ADD POW SUB x2 x1 2 POW SUB y2 y1 2
types : OP_DIV OP_MUL OP_MUL VAR VAR VAR OP_ADD OP_POW OP_SUB VAR VAR NUM OP_POW OP_SUB VAR VAR NUM


In [16]:
def preprocess_feynman_data(csv_path: str) -> list:
    df = pd.read_csv(csv_path, encoding='utf-8-sig', skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]

    results = []
    for _, row in df.iterrows():
        formula = str(row.get('Formula', '')).strip()
        if not formula or formula == 'nan':
            continue

        filename = str(row.get('Filename', '')).strip()

        # extract variables and ranges
        variables = []
        ranges = []
        for i in range(1, 11):
            name_col = f'v{i}_name'
            if name_col not in df.columns:
                break
            var_name = str(row[name_col]).strip()
            if not var_name or var_name == 'nan':
                break
            variables.append(var_name)
            ranges.append((float(row[f'v{i}_low']), float(row[f'v{i}_high'])))

        # ** → ^
        expr_with_caret = formula.replace('**', '^')

        # infix with word names (+ → ADD, * → MUL, ^ → POW, etc.)
        expr_infix = infix_to_named(expr_with_caret)

        # infix → prefix (also uses word names)
        expr_prefix = infix_to_prefix(expr_with_caret)

        # classify each prefix token: OP_ADD, OP_MUL, FUNC, NUM, CONST, VAR
        token_types = classify_prefix_tokens(expr_prefix)

        results.append({
            'filename':          filename,
            'variables':         variables,
            'ranges':            ranges,
            'expression_infix':  expr_infix,
            'expression_prefix': expr_prefix,
            'token_types':       token_types,
        })
    return results


# --- Run on ALL equations ---

CSV_IN  = "../data/FeynmanEquations (1).csv"
CSV_OUT = "../data/feynman_preprocessed.csv"

data = preprocess_feynman_data(CSV_IN)

print(f"Successfully preprocessed {len(data)} equations!\n")

# show first 5
for entry in data[:5]:
    print(f"[{entry['filename']}]")
    print(f"  prefix : {entry['expression_prefix']}")
    print(f"  types  : {entry['token_types']}")
    print()

# save to CSV
rows = []
for entry in data:
    rows.append({
        'filename':          entry['filename'],
        'variables':         str(entry['variables']),
        'ranges':            str(entry['ranges']),
        'expression_infix':  entry['expression_infix'],
        'expression_prefix': entry['expression_prefix'],
        'token_types':       entry['token_types'],
    })

out_df = pd.DataFrame(rows)
out_df.to_csv(CSV_OUT, index=False)
print(f"Saved to {CSV_OUT}")
out_df.head()

Successfully preprocessed 100 equations!

[I.6.2a]
  prefix : DIV exp DIV POW NEG theta 2 2 sqrt MUL 2 pi
  types  : OP_DIV FUNC OP_DIV OP_POW OP_NEG VAR NUM NUM FUNC OP_MUL NUM CONST

[I.6.2]
  prefix : DIV exp DIV POW NEG DIV theta sigma 2 2 MUL sqrt MUL 2 pi sigma
  types  : OP_DIV FUNC OP_DIV OP_POW OP_NEG OP_DIV VAR VAR NUM NUM OP_MUL FUNC OP_MUL NUM CONST VAR

[I.6.2b]
  prefix : DIV exp DIV POW NEG DIV SUB theta theta1 sigma 2 2 MUL sqrt MUL 2 pi sigma
  types  : OP_DIV FUNC OP_DIV OP_POW OP_NEG OP_DIV OP_SUB VAR VAR VAR NUM NUM OP_MUL FUNC OP_MUL NUM CONST VAR

[I.8.14]
  prefix : sqrt ADD POW SUB x2 x1 2 POW SUB y2 y1 2
  types  : FUNC OP_ADD OP_POW OP_SUB VAR VAR NUM OP_POW OP_SUB VAR VAR NUM

[I.9.18]
  prefix : DIV MUL MUL G m1 m2 ADD ADD POW SUB x2 x1 2 POW SUB y2 y1 2 POW SUB z2 z1 2
  types  : OP_DIV OP_MUL OP_MUL VAR VAR VAR OP_ADD OP_ADD OP_POW OP_SUB VAR VAR NUM OP_POW OP_SUB VAR VAR NUM OP_POW OP_SUB VAR VAR NUM

Saved to ../data/feynman_preprocessed.csv


,filename,variables,ranges,expression_infix,expression_prefix,token_types
0,I.6.2a,['theta'],"[(1.0, 3.0)]",exp ( SUB theta POW 2 DIV 2 ) DIV sqrt ( 2 MUL...,DIV exp DIV POW NEG theta 2 2 sqrt MUL 2 pi,OP_DIV FUNC OP_DIV OP_POW OP_NEG VAR NUM NUM F...
1,I.6.2,"['sigma', 'theta']","[(1.0, 3.0), (1.0, 3.0)]",exp ( SUB ( theta DIV sigma ) POW 2 DIV 2 ) DI...,DIV exp DIV POW NEG DIV theta sigma 2 2 MUL sq...,OP_DIV FUNC OP_DIV OP_POW OP_NEG OP_DIV VAR VA...
2,I.6.2b,"['sigma', 'theta', 'theta1']","[(1.0, 3.0), (1.0, 3.0), (1.0, 3.0)]",exp ( SUB ( ( theta SUB theta1 ) DIV sigma ) P...,DIV exp DIV POW NEG DIV SUB theta theta1 sigma...,OP_DIV FUNC OP_DIV OP_POW OP_NEG OP_DIV OP_SUB...
3,I.8.14,"['x1', 'x2', 'y1', 'y2']","[(1.0, 5.0), (1.0, 5.0), (1.0, 5.0), (1.0, 5.0)]",sqrt ( ( x2 SUB x1 ) POW 2 ADD ( y2 SUB y1 ) P...,sqrt ADD POW SUB x2 x1 2 POW SUB y2 y1 2,FUNC OP_ADD OP_POW OP_SUB VAR VAR NUM OP_POW O...
4,I.9.18,"['m1', 'm2', 'G', 'x1', 'x2', 'y1', 'y2', 'z1'...","[(1.0, 2.0), (1.0, 2.0), (1.0, 2.0), (3.0, 4.0...",G MUL m1 MUL m2 DIV ( ( x2 SUB x1 ) POW 2 ADD ...,DIV MUL MUL G m1 m2 ADD ADD POW SUB x2 x1 2 PO...,OP_DIV OP_MUL OP_MUL VAR VAR VAR OP_ADD OP_ADD...
